In [1]:
%load_ext autoreload
%autoreload 2
%cd /home/abraham/uni/ikt453/project/v1

/home/abraham/uni/ikt453/project/v1


In [5]:
from src.utils import disk
from src.utils import debug

In [4]:
league_hiarchy = 'data/samples/leaugeHiearchy.json'
boxscores = 'data/samples_nbaapi/boxscores.json'

lh = disk.read_json(league_hiarchy)
bs = disk.read_json(boxscores)

In [ ]:
from uuid import uuid4
from itertools import chain

def construct_team_dimension(lh: dict, preserve: tuple[str] = None) -> tuple[list, list, list, dict]:
    preserve = preserve or ()

    # ------------------------
    # Helper function to explode values on multiple delimiters and preserve certain values
    # ------------------------    
    def explode(output_key: str, key: str, obj: dict, **kwargs):
        split_on = (',', ' and ', '&')
        value = obj.get(key, '')
        if not value:
            return []
        
        if value in preserve or not isinstance(value, str):
            return [dict(**kwargs, **{output_key: value})]
        
        values: list[str] = [[value]]
        for splitter in split_on:
            values = chain.from_iterable(values)
            values = list(map(lambda v: v.split(splitter), values))
        
        values = chain.from_iterable(values)
        values = map(lambda v: v.strip(), values)
        values = filter(lambda v: v, values)
        return list(map(lambda v: dict(**kwargs, **{output_key: v}), values))

    # ------------------------
    # Main logic to construct teams, team roles, and team facts
    # ------------------------
    teams = []
    roles = []
    facts = []
    tc2id = {} # TODO: useful to map nba_api and sportsradar team codes to our generated team ids

    conferences = lh['conferences']
    for conference in conferences:
        c_id = str(uuid4())

        for division in conference['divisions']:
            d_id = str(uuid4())

            for team in division['teams']:
                t_id = str(uuid4())
                
                tc2id[team['alias']] = t_id

                venue = team['venue']

                meta = dict(
                    team_id=t_id,
                    team_name=team['name'],
                    team_alias=team['alias'],
                )

                teams.append(dict(
                    **meta,
                    team_market=team['market'],
                    division_id=d_id,
                    division_name=division['name'],
                    division_alias=division['alias'],
                    conference_id=c_id,
                    conference_name=conference['name'],
                    conference_alias=conference['alias'],
                    founded_in=team['founded'],
                    venue_name=venue['name'],
                    venue_capacity=venue['capacity'],
                    venue_address=venue['address'],
                    venue_city=venue['city'],
                    venue_state=venue['state'],
                    venue_zip=venue['zip'],
                    venue_country=venue['country'],
                    team_id_sportsradar=team['id'],
                    team_id_nba_api=None,
                ))


                roles.extend(chain.from_iterable(map(
                    lambda role: explode('name', role, team, **meta, role=role), 
                    [
                        'mascot',
                        'sponsor',
                        'owner',
                        'general_manager',
                    ]
                )))
    
                facts.extend(chain.from_iterable(map(
                    lambda fact: explode('value', fact, team, **meta, fact=fact),
                    [
                        'championships_won',
                        'championship_seasons',
                        'playoff_appearances',
                        'conference_titles',
                        'division_titles',
                    ]
                )))


    return teams, roles, facts, tc2id

In [ ]:
teams, team_roles, team_facts, team_code_to_id = construct_team_dimension(lh, preserve=("Kroenke Sports & Entertainment", ))

In [102]:
len(teams), len(team_roles), len(team_facts), len(team_code_to_id)

(30, 125, 180, 30)

In [103]:
import pandas as pd
df_teams = pd.DataFrame(teams)
df_roles = pd.DataFrame(team_roles)
df_facts = pd.DataFrame(team_facts)

In [104]:
# debug.prettyprint(bs, ensure_ascii=False)

In [1]:
30000/60

500.0

In [105]:
df_teams.head(3)

,team_id,team_name,team_alias,team_market,division_id,division_name,division_alias,conference_id,conference_name,conference_alias,founded_in,venue_name,venue_capacity,venue_address,venue_city,venue_state,venue_zip,venue_country,team_id_sportsradar,team_id_nba_api
0,014cfd75-a9f7-437b-bc33-eac628d1a6ca,Wizards,WAS,Washington,73244cc2-b420-4add-ad32-d725ccd27605,Southeast,SOUTHEAST,7925a8bb-bb40-46a6-98bf-8418633b5cb2,EASTERN CONFERENCE,EASTERN,1961,Capital One Arena,20356,601 F Street NW,Washington,DC,20004,USA,583ec8d4-fb46-11e1-82cb-f4ce4684ea4c,None
1,ffa63969-f978-4178-9335-31313250c80a,Hornets,CHA,Charlotte,73244cc2-b420-4add-ad32-d725ccd27605,Southeast,SOUTHEAST,7925a8bb-bb40-46a6-98bf-8418633b5cb2,EASTERN CONFERENCE,EASTERN,1988,Spectrum Center,19077,330 E. Trade Street,Charlotte,NC,28202,USA,583ec97e-fb46-11e1-82cb-f4ce4684ea4c,None
2,4f272e1b-4c08-4d88-aa2d-1e6b1b0981b7,Hawks,ATL,Atlanta,73244cc2-b420-4add-ad32-d725ccd27605,Southeast,SOUTHEAST,7925a8bb-bb40-46a6-98bf-8418633b5cb2,EASTERN CONFERENCE,EASTERN,1946,State Farm Arena,18118,One Philips Drive,Atlanta,GA,30303,USA,583ecb8f-fb46-11e1-82cb-f4ce4684ea4c,None


In [106]:
df_roles.head(3)

,team_id,team_name,team_alias,role,name
0,014cfd75-a9f7-437b-bc33-eac628d1a6ca,Wizards,WAS,mascot,G-Wiz
1,014cfd75-a9f7-437b-bc33-eac628d1a6ca,Wizards,WAS,sponsor,Robinhood
2,014cfd75-a9f7-437b-bc33-eac628d1a6ca,Wizards,WAS,owner,Ted Leonsis


In [107]:
df_facts.head(3)

,team_id,team_name,team_alias,fact,value
0,014cfd75-a9f7-437b-bc33-eac628d1a6ca,Wizards,WAS,championships_won,1
1,014cfd75-a9f7-437b-bc33-eac628d1a6ca,Wizards,WAS,championship_seasons,1978
2,014cfd75-a9f7-437b-bc33-eac628d1a6ca,Wizards,WAS,playoff_appearances,30
